# 02 — Evaluation pipeline

Runs `run_pipeline_eval` on the 108 hand-written synthetic tickets (`data/eval/queries.jsonl`). This is the **offline fallback**; the Zero-shot Claude / DistilBERT-only / Hybrid rows need an API key and the `ml` extra (`python -m eval.run --systems ...`).

In [1]:
import asyncio, json, os
from pathlib import Path
os.environ.pop("ANTHROPIC_API_KEY", None)          # offline: deterministic fallback
os.environ["USE_LOCAL_CLASSIFIER"] = "false"
from src.config import Settings
from src.api.schemas import TicketIn
from src.eval.runner import build_offline_pipeline, run_pipeline_eval, load_cases, format_report
settings = Settings(anthropic_api_key=None, use_local_classifier=False)
pipeline = asyncio.run(build_offline_pipeline(settings))
print("classifier:", pipeline.classifier.name, "| llm_enabled:", pipeline.llm_enabled)

classifier: keyword_heuristic | llm_enabled: False


In [2]:
cases = load_cases()
report = asyncio.run(run_pipeline_eval(pipeline, cases))
print("\n".join(format_report(report).splitlines()[:14]))

SUPPORT TRIAGE EVAL  system=offline_fallback  n=108  llm_used=False
classifier=keyword_heuristic  encoder=cached(deterministic_hash_256)
Macro-F1 (27 intents):   0.948
Intent top-1 / top-3:    93.5% / 93.5%
Decision agreement:      49.1%
Auto-resolve rate:       0.0%
False escalation rate:   17.9%
False auto-resolve rate: 0.0%
Priority band match:     86.1%
Latency p50 / p95 (ms):  3 / 3
Cost per ticket (USD):   0.000000



## Per-intent accuracy

Where the keyword fallback misses (synonyms and articles it does not list); see `docs/error_analysis.md`.

In [3]:
ranked = sorted(report.per_intent.items(), key=lambda kv: kv[1]["intent_accuracy"])
for intent, b in ranked:
    if b["intent_accuracy"] < 1:
        print(f"{intent:<26} n={int(b['n'])} intent_acc={b['intent_accuracy']:.2f} "
              f"decision_acc={b['decision_accuracy']:.2f}")

get_invoice                n=4 intent_acc=0.50 decision_acc=0.00
payment_issue              n=4 intent_acc=0.50 decision_acc=0.25
get_refund                 n=4 intent_acc=0.75 decision_acc=1.00
create_account             n=4 intent_acc=0.75 decision_acc=0.75
change_shipping_address    n=4 intent_acc=0.75 decision_acc=0.75
